# QUMO GPU benchmark on AMD GPUs (ROCm, any Jupyter)

Same benchmark, runner and output format as the Colab notebook, for AMD GPUs. Colab has no AMD GPUs, so run
this on any Jupyter server with ROCm:

* **AMD Developer Cloud** (GPU droplets, MI300X 1x / 8x): pick the ROCm image (JupyterLab on the base OS) or a
  Quick Start image (JupyterLab in its container), open JupyterLab, upload this notebook, or
  `git clone https://github.com/ANFAIA/QUMO` and open `examples/colab/QUMO_GPU_Benchmark_AMD.ipynb`.
* **Azure ND MI300X v5**, other MI300X / MI325X / MI355X clouds, or a local ROCm box
  (`pip install jupyterlab`, then `jupyter lab`).

Requirements (Mojo/MAX 26.6): Linux with glibc >= 2.34, the AMD GPU driver / ROCm >= 6.3.3 (>= 7.0 for MI355X),
`git`, `make`, a C compiler. Supported AMD GPUs: MI355X (`gfx950`), MI300X/MI325X (`gfx942`), MI250X (`gfx90a`),
Radeon RX 9000 / 7000 / 6900 and Ryzen AI iGPUs (`gfx1030`-`gfx1201`). Others are skipped with a recorded reason.

| engine | device | precision |
|---|---|---|
| QUMO (Mojo, MAX vendor-neutral GPU API) | CPU, GPU | CPU fp64; GPU fp32 and fp64 |
| Qiskit Aer | CPU | fp32 and fp64 |
| PennyLane Lightning | CPU (`lightning.qubit`), GPU (`lightning.amdgpu`, ROCm wheels) | fp64 / fp32 and fp64 |

CUDA-Q and `qiskit-aer-gpu` are CUDA-only, so they are recorded as *not applicable* here; `lightning.amdgpu` is
the AMD-native GPU baseline. No Colab-only API is used: every cell is plain Python, and the settings are plain
variables at the top of each cell. Run the cells top to bottom (rough estimate: 30-60 min on one MI300X, most of
it builds and the CPU reference runs).

In [ ]:
# 0. Helpers (run first)
# Plain Python on purpose (no "!" shell magics), so every cell also runs in
# Jupyter / JupyterLab / VS Code and under nbclient.
import glob, json, os, re, shlex, shutil, subprocess, sys, time

REPO_URL = "https://github.com/ANFAIA/QUMO"


def sh(cmd, check=True, cwd=None):
    """Run a shell command, stream its output, return it; raise on failure if check."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace")
    out = []
    for line in p.stdout:
        if "Crashpad" not in line:
            print(line, end="", flush=True)
        out.append(line)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return "".join(out)


def find_checkout(start):
    """A QUMO checkout containing `start` (Makefile + qumo/ + bench/run_benchmarks.py), else None."""
    d = os.path.abspath(start)
    while True:
        if all(os.path.exists(os.path.join(d, x)) for x in ("Makefile", "qumo", "bench/run_benchmarks.py")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent


def show_report(result_jsons, stem):
    """Tables + plots for result JSONs via examples/colab/compare_results.py, run in the
    benchmark venv (it has matplotlib), so the notebook kernel needs nothing extra."""
    from IPython.display import Image, Markdown, display
    md, png = stem + "_report.md", stem + "_time_vs_qubits.png"
    # MPLBACKEND=Agg: the kernel exports its inline backend, which the bench venv lacks
    sh(f"MPLBACKEND=Agg bench/.venv/bin/python examples/colab/compare_results.py {' '.join(map(shlex.quote, result_jsons))} "
       f"--out {shlex.quote(md)} --plot {shlex.quote(png)}")
    with open(md) as fh:
        text = "\n".join(ln for ln in fh.read().splitlines() if not ln.startswith("!["))  # images shown below
    display(Markdown(text))
    for p in (png, png[:-4] + "_qumo_gpu.png"):
        if os.path.exists(p):
            display(Image(filename=p))
    return [p for p in (md, png, png[:-4] + "_qumo_gpu.png") if os.path.exists(p)]


EXISTING = find_checkout(os.getcwd())
if "WORKDIR" not in globals():  # keep the first choice when this cell is re-run after step 2's chdir
    # clone, baseline venvs and archives go here; inside a checkout: its (git-ignored) build/ folder
    WORKDIR = os.path.join(EXISTING, "build", "notebook") if EXISTING else os.path.expanduser("~/qumo-bench")
os.makedirs(WORKDIR, exist_ok=True)
REPO_DIR = EXISTING or os.path.join(WORKDIR, "QUMO")
print(f"repository directory: {REPO_DIR}" + ("  (existing checkout)" if EXISTING else ""))

## 1. GPU and configuration

`CONFIG = "auto"` lets the runner pick the preset in step 2: `amd_mi300x` for MI300X/MI325X/MI350X/MI355X (or
any `gfx942`/`gfx950` with >= 64 GiB), `amd_generic` for other AMD GPUs, `cpu_only` without a GPU.

In [ ]:
# 1. Check the GPU
CONFIG = "auto"  # one of: auto, amd_mi300x, amd_generic, cpu_only, quick
QUICK = False    # True: tiny sizes (16-20 qubits for every engine), 1 repetition (smoke test)
SUITES = ""      # optional, e.g. "sv_gpu,baseline_lightning": run (and build) only these suites


def run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=120).stdout
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return ""


AGENTS = re.findall(r"Name:\s+(gfx\w+)\s*\n.*?Marketing Name:\s+(.+?)\s*\n", run(["rocminfo"]), re.S)
GPU = {"arch": AGENTS[0][0], "name": AGENTS[0][1]} if AGENTS else None
for tool in (["rocm-smi", "--showproductname", "--showmeminfo", "vram"], ["amd-smi", "list"]):
    out = run(tool)
    if out.strip():
        print("$ " + " ".join(tool) + "\n" + "\n".join(out.strip().splitlines()[:40]))
        break
if GPU:
    print(f"\nGPU 0: {GPU['name']} ({GPU['arch']}); {len(AGENTS)} GPU agent(s) in rocminfo")
else:
    print("No AMD GPU found by rocminfo. The run falls back to the CPU benchmarks and CPU baselines.")
ver = (open("/opt/rocm/.info/version").read().strip() if os.path.exists("/opt/rocm/.info/version") else "?")
print(f"ROCm: {ver}")
print(f"requested configuration: {CONFIG}" + ("  (resolved in step 2)" if CONFIG == "auto" else "")
      + ("  --quick" if QUICK else "") + (f"  --suites {SUITES}" if SUITES else ""))

## 2. Download QUMO

Default: latest `main`. Set `COMMIT` to benchmark an exact version. Re-running this step after changing
`BRANCH` / `COMMIT` moves the notebook's clone to that version (the builds in step 4 then refresh).

In [ ]:
# 2. Download QUMO and resolve the preset
BRANCH = "main"
COMMIT = ""

OWN_CLONE = os.path.abspath(REPO_DIR) == os.path.abspath(os.path.join(WORKDIR, "QUMO"))
if OWN_CLONE and os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # this notebook's own clone (a re-run): move it to the requested BRANCH / COMMIT
    if COMMIT:  # any commit, also an abbreviated hash: fetch the branches (full history), then check it out
        unshallow = "--unshallow " if os.path.exists(os.path.join(REPO_DIR, ".git", "shallow")) else ""
        sh(f"git -C {REPO_DIR} fetch -q {unshallow}origin '+refs/heads/*:refs/remotes/origin/*'")
        sh(f"git -C {REPO_DIR} checkout -q --force {shlex.quote(COMMIT)}")
    else:
        sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {shlex.quote(BRANCH)}")
        sh(f"git -C {REPO_DIR} checkout -q --force FETCH_HEAD")
elif EXISTING:
    print(f"using your checkout {REPO_DIR} as it is: BRANCH / COMMIT only move the notebook's own clone "
          "(switch your checkout with git yourself)")
else:
    os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
    if COMMIT:
        sh(f"git clone --filter=blob:none {REPO_URL} {REPO_DIR}")
        sh(f"git -C {REPO_DIR} checkout {shlex.quote(COMMIT)}")
    else:
        sh(f"git clone --depth 1 --branch {shlex.quote(BRANCH)} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git log -1 --format='%H %cd %s' --date=iso")
if not os.path.exists("bench/run_benchmarks.py"):
    raise RuntimeError("QM-E960: this QUMO version has no bench/run_benchmarks.py. Hint: use a newer BRANCH / COMMIT.")
# The runner picks the preset (one rule for every vendor; stdlib only, so the kernel's Python is enough).
sel = subprocess.run([sys.executable, "bench/run_benchmarks.py", "--select-config", "auto"],
                     capture_output=True, text=True)
lines = sel.stdout.splitlines()
if sel.returncode != 0 or not lines:
    raise RuntimeError("QM-E960: preset selection failed:\n" + sel.stdout + sel.stderr)
AUTO_CONFIG = lines[0].strip()
print("\n".join(lines[1:]))
CONFIG_NAME = AUTO_CONFIG if CONFIG == "auto" else CONFIG
CONFIG_PATH = f"bench/configs/{CONFIG_NAME}.json"
if not os.path.exists(CONFIG_PATH):
    raise RuntimeError(f"QM-E960: {CONFIG_PATH} not found in this QUMO version. Hint: pick a preset from bench/configs/.")
print(f"configuration: {CONFIG_NAME}" + ("  (auto)" if CONFIG == "auto" else f"  (forced; auto would be {AUTO_CONFIG})")
      + ("  --quick" if QUICK else ""))
print(json.load(open(CONFIG_PATH))["description"])


## 3. Install

Mojo + MAX from PyPI into `.venv` (pinned; empty = latest), the benchmark venv, and PennyLane Lightning in its own
venv: `pennylane-lightning-amdgpu` has prebuilt wheels for ROCm 7.0 and MI300-series GPUs (Linux x86_64); on other
GPUs or ROCm versions it may fail to install or to load, and the suite then records a skip.

In [ ]:
# 3. Install the Mojo/MAX toolchain and the baselines
MOJO_VERSION = "1.1.0"
MAX_VERSION = "26.6.0"
INSTALL_LIGHTNING_AMDGPU = True  # PennyLane lightning.amdgpu (GPU baseline)
INSTALL_LIGHTNING_CPU = True     # PennyLane lightning.qubit (CPU baseline)
VENVS = os.path.join(WORKDIR, "venvs")

if not shutil.which("uv"):
    sh(f"{sys.executable} -m pip install -q uv")
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
# Mojo + MAX wheels (manylinux_2_34 x86_64 / aarch64: glibc >= 2.34) into ./.venv, pinned for reproducibility
if not os.path.exists(".venv/bin/mojo"):
    if MOJO_VERSION and MAX_VERSION:
        sh("uv venv -q .venv --python 3.12")
        sh(f"uv pip install -q --python .venv/bin/python mojo=={MOJO_VERSION} "
           f"max-mojo-libs=={MAX_VERSION} max-core=={MAX_VERSION}")
    else:
        sh("make setup")  # latest releases
sh(".venv/bin/mojo --version")
# Benchmark venv: QUMO's Python tooling + the CPU baselines (Aer CPU, TKET for the compiler subset, Stim, PyMatching)
if not os.path.exists("bench/.venv/bin/python"):
    sh("uv venv -q bench/.venv --python 3.12")
sh("uv pip install -q --python bench/.venv/bin/python qiskit qiskit-aer pytket pytket-qiskit stim pymatching "
   "numpy matplotlib psutil")


def baseline_venv(name, packages, env_var):
    """Own venv per GPU baseline (their CUDA / ROCm wheels and Qiskit pins must not clash).
    A failed install is reported and the suite records a skip; the notebook continues."""
    py = os.path.join(VENVS, name, "bin", "python")
    try:
        if not os.path.exists(py):
            sh(f"uv venv -q {os.path.join(VENVS, name)} --python 3.12")
        sh(f"uv pip install -q --python {py} {packages}")
        os.environ[env_var] = py
    except RuntimeError as e:
        print(f"NOTE: baseline '{name}' not installed ({e}); its suite will record a skip.")


pkgs = ["pennylane", "numpy"]
if INSTALL_LIGHTNING_CPU or INSTALL_LIGHTNING_AMDGPU:
    pkgs.append("pennylane-lightning")
if INSTALL_LIGHTNING_AMDGPU and GPU:
    pkgs.append("pennylane-lightning-amdgpu")
if len(pkgs) > 2:
    baseline_venv("lightning", " ".join(pkgs), "QUMO_LIGHTNING_PYTHON")
print({k: v for k, v in os.environ.items() if k.startswith("QUMO_")})

## 4. Build

The runner compiles what the preset (or `SUITES`) needs (CPU benchmark, the GPU benchmark for fp64 and fp32 for this GPU's
`gfx` target, the Python extension). A failing build is printed and recorded, not fatal.

In [ ]:
# 4. Build QUMO (benchmark binaries incl. GPU)
BUILD_CLI = False  # the qumo CLI + MCP server (not needed by the benchmark)

t0 = time.time()
if BUILD_CLI:
    sh("make cli-gpu" if GPU else "make cli")
build_args = ["--config", CONFIG_PATH, "--build-only"] + (["--suites", SUITES] if SUITES else [])
build_args += ["--quick"] if QUICK else []  # a quick run builds no diagnostic variants
sh("bench/.venv/bin/python bench/run_benchmarks.py " + " ".join(shlex.quote(a) for a in build_args), check=False)
print(f"build time: {time.time() - t0:.0f} s")

## 5. Run

In [ ]:
# 5. Run the benchmark
RUN_LABEL = ""  # optional free text stored in the result (no personal data)

args = ["--config", CONFIG_PATH, "--out", "bench/results/cloud/"]
args += ["--quick"] if QUICK else []
args += ["--suites", SUITES] if SUITES else []
args += ["--label", RUN_LABEL] if RUN_LABEL else []
log = sh("bench/.venv/bin/python bench/run_benchmarks.py " + " ".join(shlex.quote(a) for a in args), check=False)
found = re.findall(r"^results: (\S+\.json)\s*$", log, re.M)
RESULT_JSON = found[-1] if found else max(glob.glob("bench/results/cloud/*.json"), key=os.path.getmtime)
print("result file:", RESULT_JSON)

## 6. Results

In [ ]:
# 6. Tables and plots
REPORT_FILES = show_report([RESULT_JSON], RESULT_JSON[:-5])

## 7. Save and compare

In [ ]:
# 7. Save the results; optionally compare with other runs (NVIDIA / Apple / other AMD GPUs)
OTHER_RESULTS = []  # e.g. ["~/results/qumo_bench_colab_t4_....json"]: any QUMO result JSONs copied here
import zipfile  # noqa: E402

stem = RESULT_JSON[:-5]
files_to_save = sorted(f for f in glob.glob(stem + "*") if os.path.isfile(f))
ZIP = os.path.join(WORKDIR, os.path.basename(stem) + ".zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_save:
        z.write(f, os.path.basename(f))
print("archive:", ZIP, "\n  " + "\n  ".join(os.path.basename(f) for f in files_to_save))
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(ZIP)))
except Exception:
    pass
print("Copy it off the machine before deleting the VM, e.g.:  scp <user>@<host>:" + ZIP + " .")
others = [os.path.expanduser(p) for p in OTHER_RESULTS if os.path.exists(os.path.expanduser(p))]
if others:
    COMPARE_FILES = show_report([RESULT_JSON] + others, os.path.join(WORKDIR, "qumo_compare"))

## Notes

* GPUs outside the Mojo/MAX support list are **skipped with a recorded reason**; everything CPU-side still runs.
  `rocminfo | grep gfx` shows the architecture.
* MI300X has full-rate fp64, so QUMO GPU fp64 vs Lightning AMDGPU fp64 is the fair headline comparison here;
  fp32 is compared with fp32.
* The run records the ROCm version, GPU arch and memory, CPU, RAM, commit, a hash of QUMO's GPU engine sources
  and load average in the JSON; every row records what its timed region contains and what the engine fused. The
  report's *How each series was timed* section lists them and refuses to put QUMO GPU rows of different engines in
  one cross-run table. Widths of 20 qubits and fewer are marked: fixed per-call overhead dominates there.
* `sv_gpu_probe` (off in the AMD presets; set `"enabled": true` or `SUITES="sv_gpu_probe"`) runs the cost probe of
  QUMO's GPU engine when the QUMO version has one: sweep floor, per-op costs, transposes, stream bandwidth.
* Compare with Colab / NVIDIA / Apple runs: `bench/.venv/bin/python examples/colab/compare_results.py *.json --out
  compare.md --plot compare.png`; share via a GitHub issue or a pull request to `bench/results/cloud/`. The JSON
  stores no host names, user paths or credentials.